In [57]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from rank_bm25 import BM25Okapi
import pickle
import json 
import re
import os
import numpy as np
import uuid


In [58]:
query = """What technique has been used to prove the specific, causal role of the precuneus in self-reference?"""

In [59]:
model = SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [60]:
query_embeddings = model.encode(
                            query,
                            normalize_embeddings=True
                        )
query_embeddings

array([-9.51771345e-03,  2.13416871e-02, -2.21215957e-03, -5.40507883e-02,
        6.65024817e-02,  5.62983751e-02,  5.36923185e-02,  2.67389845e-02,
       -3.85781825e-02, -6.83054700e-02, -1.59209426e-02,  1.56548023e-02,
       -2.50258353e-02, -1.14261657e-02, -1.63186435e-02,  4.24657799e-02,
        5.30340597e-02, -1.11758867e-02, -9.57988482e-03, -2.11106008e-03,
        6.47653267e-03,  7.98119325e-03,  2.59260094e-04,  2.33572759e-02,
        1.11515098e-03, -1.04210293e-02,  2.33596694e-02,  1.35504967e-02,
       -9.04469285e-03,  7.70165119e-03,  2.61281151e-02, -2.38251057e-03,
       -4.70067821e-02, -5.36043458e-02,  3.31481639e-03,  2.92518362e-03,
        1.91337671e-02, -1.03309155e-02, -8.13451558e-02, -2.03524111e-03,
       -6.41100407e-02, -7.09997909e-03, -1.06695276e-02, -3.14170234e-02,
       -7.65826628e-02, -1.04490919e-02, -6.66637570e-02,  3.80154476e-02,
       -2.40862183e-02,  2.54042111e-02, -4.75568920e-02,  2.86058560e-02,
        2.07959171e-02,  

In [61]:
type(query_embeddings)

numpy.ndarray

In [62]:
# check from qdrant-client
client = QdrantClient(host = "localhost", port=6333)

search_results = client.query_points(
    collection_name="sample_docs_collections",
    query=query_embeddings,
    limit=10,    
).points

In [63]:
for point in search_results:
    print(f"Score: {point.score:.4f} | id: {point.id}")


Score: 0.7569 | id: 691ea00c-3713-51c2-b63d-4e2150342b72
Score: 0.7295 | id: 97d13dac-a41f-52c4-b138-46185271ab2c
Score: 0.7169 | id: e4f0ba69-6fef-50f0-aad1-91d87eacba86
Score: 0.7159 | id: 783e391f-e153-5d51-b3f1-aa4634141dcb
Score: 0.7152 | id: e469bbc2-88fc-52b3-aa3c-3eb6047082b5
Score: 0.7147 | id: 931662b0-a60f-592f-a205-617b2454ea57
Score: 0.7112 | id: e98bb74e-cf90-53c9-99b7-dd6a70926cfc
Score: 0.7111 | id: 3b7e9e8e-6a4c-56ed-966e-16d123518c9d
Score: 0.7068 | id: 5a4fa22b-a3a5-5776-86fb-a12bcfa4b276
Score: 0.7038 | id: 685172eb-1632-5dfa-819c-a5eb7a45f23c


In [64]:
# similar with finding from build_bm25_index

def tokenise_query_text(text):
    clean_text = re.sub(r'[^\w\s]', ' ', text.lower())
    return clean_text.split()
tokenise_query = tokenise_query_text(query)
tokenise_query

['what',
 'technique',
 'has',
 'been',
 'used',
 'to',
 'prove',
 'the',
 'specific',
 'causal',
 'role',
 'of',
 'the',
 'precuneus',
 'in',
 'self',
 'reference']

In [65]:
with open("../../data/sample_bm25_index.pkl" , "rb") as f_bm25:
    sample_bm25 = pickle.load(f_bm25)

with open("../../data/sample_mapping_table.pkl" , "rb") as f_map:
    sample_mapping_table = pickle.load(f_map)

In [66]:
doc_scores = sample_bm25.get_scores(tokenise_query)
top_indices = np.argsort(doc_scores)[::-1][:10]
top_indices



array([105, 104, 102,  90,  96,  12, 101,  99,  92,  91])

In [67]:
results = []
for rank, idx in enumerate(top_indices, start=1):
        score = doc_scores[idx]
        if score == 0:
            continue
            
        data_mapping = sample_mapping_table[idx]
        
        results.append({
            "rank": rank,
            "score": round(score, 4),
            "uuid": data_mapping.get("chunk_id"),
            "text":data_mapping.get("text")
        })
results

[{'rank': 1,
  'score': np.float64(28.4334),
  'uuid': '22data_c21',
  'text': 'frontal and posterior cingulate/medial parietal cortices as well as\nthalamic regions,it has been demonstrated that gamma synchrony\nis a common neural event in both minimal self-reference and\nextended self-reference, where the degree of synchrony is clearly\nrelated to the degree of self-reference (Lou et al., 2010). As noted\nabove, this mechanism of gamma synchrony is not likely to be\naffected by the recent discovery that microsaccades may give rise\nto gamma oscillations. These gamma oscillations are induced by\nextra cerebral sources primarily in the frontal lobe, and have not\nbeen found in the parietal cortices (Carl et al., 2012). In addition\nwe used beamforming for source localization which allows a clear\ndistinction between sources.\nThe degree of gamma synchrony in the paralimbic network is\na potential mechanism between conscious experiences and widely\ndifferent degrees of self-reference. T

In [ ]:


# set
search_hit_scores = {str(point.id): point.score for point in search_results}

matched_results = []


for item in sample_mapping_table:
    chunk_id = item.get("chunk_id")
    
   
    valid_uuid = str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk_id))
    

    if valid_uuid in search_hit_scores:
       
        qdrant_score = search_hit_scores[valid_uuid]
        
   
        matched_results.append({
            "uuid": valid_uuid,
            "chunk_id": chunk_id,
            "qdrant_score": round(float(qdrant_score), 4), 
            "text": item.get("text")
        })

# =====================================================================
# PRINT VERIFICATION AND SORT BY SCORE
# =====================================================================
# Sort the results descending so the highest Qdrant scores show up first
matched_results = sorted(matched_results, key=lambda x: x["qdrant_score"], reverse=True)

print(f"Successfully matched and isolated {len(matched_results)} documents with scores.\n")

for rank, match in enumerate(matched_results, start=1):
    print(f"Rank #{rank} | Score: {match['qdrant_score']} | Chunk ID: {match['chunk_id']}")
    print(f"Content: {match['text'][:120]}...")
    print("-" * 50)


Successfully matched and isolated 10 documents with scores.

Rank #1 | Score: 0.7569 | Chunk ID: 22data_c15
Content: amus (Henson et al., 1999; McDermott et al., 1999; Wiggs et al.,
1999; Cabeza and Nyberg, 2000; Gardiner, 2001; Tulving,...
--------------------------------------------------
Rank #2 | Score: 0.7295 | Chunk ID: 22data_c19
Content: the subject what to do. Self-referential stimuli are thought to be
evaluated in the dorso-medial prefrontal cortex as ju...
--------------------------------------------------
Rank #3 | Score: 0.7169 | Chunk ID: 28data_c126
Content: 6 
hypothesis. Cogn Neurosci. 2011;2(2):98-113. 
7 
16. 
Blanke O, Metzinger T. Full-body illusions and minimal phenomen...
--------------------------------------------------
Rank #4 | Score: 0.7159 | Chunk ID: 22data_c16
Content: activity at that time interval after stimulus presentation is particularly
important for self-representation. Speciﬁcity...
--------------------------------------------------
Rank #5 | Scor

In [69]:
matched_results

[{'uuid': '691ea00c-3713-51c2-b63d-4e2150342b72',
  'chunk_id': '22data_c15',
  'qdrant_score': 0.7569,
  'text': 'amus (Henson et al., 1999; McDermott et al., 1999; Wiggs et al.,\n1999; Cabeza and Nyberg, 2000; Gardiner, 2001; Tulving, 2002).\nFurther hemodynamic studies have conﬁrmed that the paralimbic\nnetwork is active in self-awareness (Kjaer et al., 2002b), and single\npulse transcranial magnetic stimulation conﬁrmed the speciﬁcity\nand causality of the network in extended self-awareness (Lou et al.,\n2004; see Figure 3).\nFIGURE 3 | Causality and speciﬁcity of one node of the paralimbic\nnetwork in self-reference. The ﬁgure demonstrates the effects of applying\ntranscranial magnetic stimulation (TMS) in targeting the\nprecuneus/posterior cingulate region in a task probing self-reference. The\ncausality of this region in self-reference is seen by the fact that retrieval of\nself-judgment was signiﬁcantly less efﬁcient with TMS at a latency of\n160 ms than with a latency of 0 ms 